# Functional-module workflow

This notebook applies the module-detection and pruning workflow inspired by [Soligo et al. (2025)](https://proceedings.mlr.press/v267/soligo25a.html).

It uses a synthetic quadrant-classification task trained with cross-entropy, then imposes two weight groups to exercise the analysis. It is an API example, not the paper's MiniGrid PPO experiment.


## Experiment


In [1]:
import networkx as nx
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.weights import Pruning
from tdhook.workflow import Workflow
from xdrl import interpret

SEED = 6163
FULL_TRAINING_CONFIG = {
    "algorithm": "PPO",
    "seed": SEED,
    "training_frames": 4000000,
    "finetuning_frames": 2000000,
    "num_envs": 16,
    "rollout_steps": 128,
    "epochs": 16,
    "minibatches": 8,
    "learning_rate": 0.0005,
    "discount": 0.99,
    "gae_lambda": 0.99,
    "clip_epsilon": 0.2,
    "entropy_coefficient": 0.01,
    "max_grad_norm": 0.5,
    "hidden_size": 32,
    "hidden_layers": 2,
    "connection_cost": 0.02,
    "distance_subtraction": 0.95,
    "swap_frequency": 2,
    "regularization_warmup": [0.2, 0.3],
    "prune_fraction": 0.01,
}
AGREEMENT_TOLERANCE = 0.05
torch.manual_seed(SEED)

In [2]:
def build_dataset():
    generator = torch.Generator().manual_seed(SEED)
    observations = torch.rand(384, 2, generator=generator) * 2 - 1
    actions = (observations[:, 0] > 0).long() + 2 * (observations[:, 1] > 0).long()
    return {
        "train_observation": observations[:256],
        "train_action": actions[:256],
        "eval_observation": observations[256:],
        "eval_action": actions[256:],
        "reference_correlation_alignment_ari": None,
        "reference_intervention_effect": None,
        "axis_feature_indices": [0, 1],
        "action_axis": ["x", "x", "y", "y"],
        "environment": "compact-grid-policy",
        "training": {"steps": 120, "optimizer": "Adam", "lr": 0.03},
    }


bundle = build_dataset()
{"environment": bundle["environment"], "training": bundle["training"]}

{'environment': 'compact-grid-policy',
 'training': {'steps': 120, 'optimizer': 'Adam', 'lr': 0.03}}

## Train the classifier


In [3]:
input_dim = int(bundle["eval_observation"].shape[-1])
hidden_size = 32 if "smoke" == "full" else 16
action_count = 4
policy_net = torch.nn.Sequential(
    torch.nn.Linear(input_dim, hidden_size),
    torch.nn.Tanh(),
    torch.nn.Linear(hidden_size, hidden_size),
    torch.nn.Tanh(),
    torch.nn.Linear(hidden_size, action_count, bias=False),
)


def locality_penalty(model):
    total = torch.zeros(())
    line = [
        torch.linspace(-1, 1, input_dim),
        torch.linspace(-1, 1, hidden_size),
        torch.linspace(-1, 1, hidden_size),
        torch.linspace(-1, 1, action_count),
    ]
    for layer, left, right in zip((model[0], model[2], model[4]), line, line[1:]):
        distance = (right[:, None] - left[None, :]).abs()
        total = total + torch.log1p(layer.weight.abs() * distance).mean()
    return total


optimizer = torch.optim.Adam(policy_net.parameters(), lr=bundle["training"]["lr"])
for _ in range(bundle["training"]["steps"]):
    logits = policy_net(bundle["train_observation"])
    log_sparsity = sum((torch.log1p(parameter.abs()).mean() for parameter in policy_net.parameters()))
    loss = (
        torch.nn.functional.cross_entropy(logits, bundle["train_action"])
        + 0.002 * log_sparsity
        + 0.002 * locality_penalty(policy_net)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
half = hidden_size // 2
with torch.no_grad():
    policy_net[0].weight[:half, 1] = 0
    policy_net[0].weight[half:, 0] = 0
    policy_net[2].weight[:half, half:] = 0
    policy_net[2].weight[half:, :half] = 0
    policy_net[4].weight[:2, half:] = 0
    policy_net[4].weight[2:, :half] = 0
policy_net.eval()
sparsity = (
    torch.cat([parameter.detach().flatten() for parameter in policy_net.parameters()])
    .abs()
    .lt(0.01)
    .float()
    .mean()
    .item()
)
{"near_zero_fraction": sparsity, "locality_penalty": locality_penalty(policy_net).item()}

{'near_zero_fraction': 0.4713541567325592,
 'locality_penalty': 0.37053513526916504}

## Collect activations


In [4]:
eval_batch = TensorDict(
    {"observation": bundle["eval_observation"].float()}, batch_size=[len(bundle["eval_action"])], names=["transition"]
)
policy = TensorDictModule(policy_net, in_keys=["observation"], out_keys=["logits"])
component = interpret(policy)
native_logits = component(eval_batch.clone())["logits"].detach().clone()
cache = component.run(Workflow(ActivationCaching("module.3", cache_key=("activation", "hidden"))), eval_batch.clone())
instrumented_logits = cache.data["logits"].detach()
torch.testing.assert_close(instrumented_logits, native_logits, rtol=0, atol=0)
activations = cache.data["activation", "hidden", "module.3"].detach()
{"parity": True, "examples": len(eval_batch), "activation_shape": tuple(activations.shape)}

{'parity': True, 'examples': 128, 'activation_shape': (128, 16)}

## Detect modules


In [5]:
def graph_from_similarity(similarity, neighbours=3):
    graph = nx.Graph()
    graph.add_nodes_from(range(similarity.shape[0]))
    values = similarity.abs().clone()
    values.fill_diagonal_(-1)
    for i in range(similarity.shape[0]):
        for j in values[i].topk(min(neighbours, similarity.shape[0] - 1)).indices.tolist():
            weight = float(values[i, j])
            if weight > 1e-08:
                graph.add_edge(i, j, weight=weight)
    return graph


def adjusted_rand(left, right, size):
    labels = []
    for partition in (left, right):
        label = torch.empty(size, dtype=torch.long)
        for index, community in enumerate(partition):
            label[list(community)] = index
        labels.append(label)
    same_left = labels[0][:, None] == labels[0][None, :]
    same_right = labels[1][:, None] == labels[1][None, :]
    upper = torch.triu(torch.ones(size, size, dtype=torch.bool), diagonal=1)
    tp = (same_left & same_right & upper).sum().item()
    fp = (same_left & ~same_right & upper).sum().item()
    fn = (~same_left & same_right & upper).sum().item()
    tn = (~same_left & ~same_right & upper).sum().item()
    denominator = (tp + fp) * (fp + tn) + (tp + fn) * (fn + tn)
    return 0.0 if denominator == 0 else 2 * (tp * tn - fp * fn) / denominator


assert adjusted_rand([{0, 1}, {2, 3, 4}], [{0, 1}, {2, 3, 4}], 5) == 1.0


def isolation(graph, partition):
    scores = []
    for community in partition:
        internal = sum((data["weight"] for u, v, data in graph.edges(data=True) if u in community and v in community))
        external = sum(
            (data["weight"] for u, v, data in graph.edges(data=True) if (u in community) != (v in community))
        )
        scores.append(internal / (internal + external) if internal + external else 0.0)
    return sum(scores) / len(scores)


def internal_louvain(weight_graph, input_nodes):
    internal = weight_graph.subgraph(set(weight_graph) - set(input_nodes))
    partition = [set(group) for group in nx.community.louvain_communities(internal, seed=SEED, weight="weight")]
    for node in input_nodes:
        strengths = [
            sum((weight_graph[node][neighbour]["weight"] for neighbour in weight_graph[node] if neighbour in group))
            for group in partition
        ]
        partition[max(range(len(strengths)), key=strengths.__getitem__)].add(node)
    return partition


def extended_louvain(weight_graph, correlation_partition, initial_partition):
    partition = [set(group) for group in initial_partition]

    def score(groups):
        return isolation(weight_graph, groups) + adjusted_rand(groups, correlation_partition, len(weight_graph))

    while len(partition) > 1:
        current = score(partition)
        best = (current, None)
        for i in range(len(partition)):
            for j in range(i + 1, len(partition)):
                if not any((weight_graph.has_edge(u, v) for u in partition[i] for v in partition[j])):
                    continue
                candidate = [group.copy() for group in partition]
                candidate[i] |= candidate[j]
                del candidate[j]
                best = max(best, (score(candidate), (i, j)), key=lambda item: item[0])
        if best[1] is None or best[0] <= current + 1e-12:
            break
        i, j = best[1]
        partition[i] |= partition[j]
        del partition[j]
    return sorted((set(group) for group in partition), key=lambda group: min(group))


layer_sizes = (input_dim, hidden_size, hidden_size, action_count)
offsets = (0, input_dim, input_dim + hidden_size, input_dim + 2 * hidden_size)
weight_graph = nx.Graph()
weight_graph.add_nodes_from(range(sum(layer_sizes)))
for layer, left_offset, right_offset in zip((policy_net[0], policy_net[2], policy_net[4]), offsets, offsets[1:]):
    for right in range(layer.weight.shape[0]):
        for left in range(layer.weight.shape[1]):
            weight = float(layer.weight[right, left].detach().abs())
            if weight > 1e-08:
                weight_graph.add_edge(left_offset + left, right_offset + right, weight=weight)
with torch.no_grad():
    hidden_one = policy_net[1](policy_net[0](eval_batch["observation"]))
    hidden_two = policy_net[3](policy_net[2](hidden_one))
all_activations = torch.cat((eval_batch["observation"], hidden_one, hidden_two, native_logits), dim=1)
correlation = torch.corrcoef(all_activations.T).nan_to_num()
correlation_graph = graph_from_similarity(correlation)
correlation_partition = [
    set(group) for group in nx.community.louvain_communities(correlation_graph, seed=SEED, weight="weight")
]
initial_partition = internal_louvain(weight_graph, range(input_dim))
modules = extended_louvain(weight_graph, correlation_partition, initial_partition)
correlation_alignment_ari = adjusted_rand(modules, correlation_partition, len(weight_graph))
{
    "modules": modules,
    "correlation_partition": correlation_partition,
    "correlation_alignment_ari": correlation_alignment_ari,
    "isolation": isolation(weight_graph, modules),
}

{'modules': [{0,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   34,
   35},
  {1, 10, 11, 12, 13, 14, 15, 16, 17, 26, 27, 28, 29, 30, 31, 32, 33, 36, 37}],
 'correlation_partition': [{0, 4, 5, 6, 8},
  {1, 11, 12, 13, 14, 15, 16, 17, 27, 32},
  {10, 26, 28, 29},
  {21, 23, 24, 25, 34, 35},
  {2, 3, 7, 9, 18, 19, 20, 22},
  {30, 31, 33, 36, 37}],
 'correlation_alignment_ari': 0.3392857142857143,
 'isolation': 1.0}

In [6]:
coordinates = eval_batch["observation"][:, bundle["axis_feature_indices"]]
axis_rows = []
hidden_two_offset = offsets[2]
for module_index, module in enumerate(modules):
    hidden_units = sorted(
        (node - hidden_two_offset for node in module if hidden_two_offset <= node < hidden_two_offset + hidden_size)
    )
    if not hidden_units:
        continue
    response = activations[:, hidden_units].mean(dim=1)
    corr = torch.corrcoef(torch.stack((response, coordinates[:, 0], coordinates[:, 1]))).nan_to_num()[0, 1:]
    axis_rows.append(
        {
            "module": module_index,
            "units": hidden_units,
            "x": float(corr[0]),
            "y": float(corr[1]),
            "selectivity": float(corr.abs().max()),
        }
    )
axis_alignment = sum((row["selectivity"] for row in axis_rows)) / len(axis_rows)
eligible_rows = [row for row in axis_rows if len(row["units"]) <= hidden_size // 2]
if not eligible_rows:
    raise RuntimeError("no module permits a disjoint size-matched null")
target_row = max(eligible_rows, key=lambda row: abs(row["x"]))
target_units = target_row["units"]
available = [unit for unit in range(hidden_size) if unit not in target_units]
if len(available) < len(target_units):
    raise RuntimeError("cannot construct a disjoint size-matched null")
control_units = available[: len(target_units)]
{
    "axis_alignment": axis_alignment,
    "modules": axis_rows,
    "selected_units": target_units,
    "control_units": control_units,
    "selection_used_eval_actions": False,
}

{'axis_alignment': 0.8304908275604248,
 'modules': [{'module': 0,
   'units': [0, 1, 2, 3, 4, 5, 6, 7],
   'x': 0.8732441067695618,
   'y': 0.122635118663311,
   'selectivity': 0.8732441067695618},
  {'module': 1,
   'units': [8, 9, 10, 11, 12, 13, 14, 15],
   'x': 0.09101299941539764,
   'y': 0.7877375483512878,
   'selectivity': 0.7877375483512878}],
 'selected_units': [0, 1, 2, 3, 4, 5, 6, 7],
 'control_units': [8, 9, 10, 11, 12, 13, 14, 15],
 'selection_used_eval_actions': False}

## Prune a module


In [7]:
def absolute_importance(*, parameter, **_):
    return parameter.abs()


def target_importance(*, parameter, **_):
    score = torch.ones_like(parameter)
    if parameter.ndim == 2:
        score[:, target_units] = 0
    return score


def control_importance(*, parameter, **_):
    score = torch.ones_like(parameter)
    if parameter.ndim == 2:
        score[:, control_units] = 0
    return score


def paired_pruning(selected, importance):
    baseline_workflow = Workflow(
        Pruning(importance_callback=absolute_importance, amount_to_prune=0, relative_path="module.4")
    )
    intervention_workflow = Workflow(
        Pruning(importance_callback=importance, amount_to_prune=action_count * len(selected), relative_path="module.4")
    )
    torch.manual_seed(SEED)
    baseline = component.run(baseline_workflow, eval_batch.clone())
    torch.manual_seed(SEED)
    intervention = component.run(intervention_workflow, eval_batch.clone())
    return {"baseline": baseline, "intervention": intervention}


target_pair = paired_pruning(target_units, target_importance)
control_pair = paired_pruning(control_units, control_importance)
for pair in (target_pair, control_pair):
    torch.testing.assert_close(pair["baseline"].data["logits"], native_logits, rtol=0, atol=0)
    assert not any((child._forward_hooks for child in policy.modules()))

In [8]:
labels = bundle["eval_action"].long()
baseline_actions = native_logits.argmax(dim=-1)
axis_ids = {
    axis: torch.tensor([index for index, value in enumerate(bundle["action_axis"]) if value == axis])
    for axis in ("x", "y")
}


def axis_frequency(actions, axis):
    return float(torch.isin(actions, axis_ids[axis]).float().mean())


def effect(pair):
    actions = pair["intervention"].data["logits"].argmax(dim=-1)
    return {
        "accuracy": float((actions == labels).float().mean()),
        "accuracy_delta": float((actions == labels).float().mean() - (baseline_actions == labels).float().mean()),
        "action_disagreement": float((actions != baseline_actions).float().mean()),
        "x_action_frequency_delta": axis_frequency(actions, "x") - axis_frequency(baseline_actions, "x"),
        "y_action_frequency_delta": axis_frequency(actions, "y") - axis_frequency(baseline_actions, "y"),
    }


metrics = {
    "correlation_alignment_ari": correlation_alignment_ari,
    "axis_alignment_diagnostic": axis_alignment,
    "baseline_accuracy": float((baseline_actions == labels).float().mean()),
    "target_intervention": effect(target_pair),
    "size_matched_control": effect(control_pair),
}
reference = bundle["reference_correlation_alignment_ari"]
agreement = None if "smoke" == "smoke" else abs(correlation_alignment_ari - float(reference)) <= AGREEMENT_TOLERANCE
metrics["reference_correlation_alignment_ari"] = reference
metrics["agreement_within_0.05"] = agreement
metrics

{'correlation_alignment_ari': 0.3392857142857143,
 'axis_alignment_diagnostic': 0.8304908275604248,
 'baseline_accuracy': 0.5390625,
 'target_intervention': {'accuracy': 0.4375,
  'accuracy_delta': -0.1015625,
  'action_disagreement': 0.640625,
  'x_action_frequency_delta': -0.5234375,
  'y_action_frequency_delta': 0.5234375},
 'size_matched_control': {'accuracy': 0.5078125,
  'accuracy_delta': -0.03125,
  'action_disagreement': 0.1875,
  'x_action_frequency_delta': 0.1875,
  'y_action_frequency_delta': -0.1875},
 'reference_correlation_alignment_ari': None,
 'agreement_within_0.05': None}

## Results


In [9]:
{
    "module_alignment_ari": correlation_alignment_ari,
    "selected_module": sorted(target_units),
    "control_module": sorted(control_units),
    "policy_accuracy": metrics["baseline_accuracy"],
    "selected_pruning": metrics["target_intervention"],
    "control_pruning": metrics["size_matched_control"],
}

{'module_alignment_ari': 0.3392857142857143,
 'selected_module': [0, 1, 2, 3, 4, 5, 6, 7],
 'control_module': [8, 9, 10, 11, 12, 13, 14, 15],
 'policy_accuracy': 0.5390625,
 'selected_pruning': {'accuracy': 0.4375,
  'accuracy_delta': -0.1015625,
  'action_disagreement': 0.640625,
  'x_action_frequency_delta': -0.5234375,
  'y_action_frequency_delta': 0.5234375},
 'control_pruning': {'accuracy': 0.5078125,
  'accuracy_delta': -0.03125,
  'action_disagreement': 0.1875,
  'x_action_frequency_delta': 0.1875,
  'y_action_frequency_delta': -0.1875}}